# Phase 03F: Teacher B BLIP Base Offline Preflight

Mục tiêu của notebook là xác minh `Salesforce/blip-image-captioning-base`
trước khi cho phép full training Teacher B. Notebook không đánh giá Held-out
Test và không phát hành checkpoint huấn luyện.

Phiên bản notebook: `phase3f-v1.3`, tương thích `transformers 5.0.0`;
BLIP processor được ép `use_fast=False` để giữ preprocessing gốc.

## Kaggle inputs bắt buộc

1. Phase 03E Teacher A aggregate/Teacher B authorization output.
2. Phase 01 formal output.
3. MS COCO 2014 image dataset.
4. BLIP Base offline model package có `MODEL_SOURCE_MANIFEST.json`.

## Runtime

- Accelerator: GPU T4 hoặc mạnh hơn.
- Internet: Off.
- Chạy tuần tự Cell 1 đến Cell 11.
- Không `Run All` trước khi Cell 3 xác minh đúng source revision.

## Leakage boundary

- Train và Validation caption được phép đọc cho preflight.
- `mscoco_test.parquet` không được mở hoặc hash.
- Held-out Test image/caption/metric không được sử dụng.


## Cell 1 — Environment và khám phá input

### Công việc

- khóa chế độ offline và deterministic;
- tạo output tree;
- tìm Phase 03E, Phase 01, MS COCO và BLIP package;
- xác nhận đúng repo/revision từ source manifest;
- chưa load model, caption hoặc Held-out Test.

### Thời gian dự kiến

10–60 giây.

### Tiến độ hiện tại

Cell 1/11 — khám phá và định danh toàn bộ input.

### Đầu ra dự kiến

```text
Status                    : PASSED
Teacher B authorization   : FOUND
BLIP offline source       : FOUND
Resolved revision         : 82a37760796d32b1411fe092ab5d4e227313294b
Test Set status           : SEALED
Progress                  : 1 / 11 cells passed
```


In [1]:
import gc
import hashlib
import json
import math
import os
import platform
import random
import statistics
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import torch
import transformers

CELL_STARTED_AT = time.perf_counter()
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = (
    Path("/kaggle/working")
    / ""
    / "phase3_teacher_b_blip_preflight"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
LOG_DIR = OUTPUT_ROOT / "logs"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT,
    AUDIT_DIR,
    CONFIG_DIR,
    LOG_DIR,
    MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

EXPECTED_REPO_ID = "Salesforce/blip-image-captioning-base"
EXPECTED_REVISION = "82a37760796d32b1411fe092ab5d4e227313294b"
SEED = 42
DEVICE = torch.device("cuda")


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with open(path, "r", encoding="utf-8") as source:
        return json.load(source)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True)
        + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as source:
        for chunk in iter(lambda: source.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def exactly_one(root, name):
    matches = sorted(Path(root).rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}, tìm thấy {len(matches)}: "
        f"{[str(path) for path in matches]}"
    )
    return matches[0].resolve()


def discover_kaggle_inputs_once():
    target_names = {
        "PHASE3_TEACHER_B_PREFLIGHT_AUTHORIZED.json",
        "PHASE1_FORMAL_DELIVERABLES_READY.json",
        "MODEL_SOURCE_MANIFEST.json",
    }
    file_matches = {name: [] for name in target_names}
    train_leaves = []
    val_leaves = []
    for current_text, directory_names, file_names in os.walk(INPUT_ROOT):
        current = Path(current_text)
        current_name = current.name
        if current_name == "train2014" and any(
            name.startswith("COCO_train2014_") and name.endswith(".jpg")
            for name in file_names
        ):
            train_leaves.append(current.resolve())
            directory_names[:] = []
            continue
        if current_name == "val2014" and any(
            name.startswith("COCO_val2014_") and name.endswith(".jpg")
            for name in file_names
        ):
            val_leaves.append(current.resolve())
            directory_names[:] = []
            continue
        present = target_names.intersection(file_names)
        for name in present:
            file_matches[name].append((current / name).resolve())
    return file_matches, sorted(set(train_leaves)), sorted(set(val_leaves))


assert INPUT_ROOT.is_dir()
assert torch.cuda.is_available(), "Phase 03F yêu cầu GPU."
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=False)
set_all_seeds(SEED)

(
    DISCOVERED_FILE_MATCHES,
    discovered_train_leaves,
    discovered_val_leaves,
) = discover_kaggle_inputs_once()

phase3e_matches = DISCOVERED_FILE_MATCHES[
    "PHASE3_TEACHER_B_PREFLIGHT_AUTHORIZED.json"
]
assert len(phase3e_matches) == 1, phase3e_matches
PHASE3E_AUTH_PATH = phase3e_matches[0]
PHASE3E_ROOT = PHASE3E_AUTH_PATH.parent
phase1_matches = DISCOVERED_FILE_MATCHES[
    "PHASE1_FORMAL_DELIVERABLES_READY.json"
]
assert len(phase1_matches) == 1, phase1_matches
PHASE1_READY_PATH = phase1_matches[0]
PHASE1_ROOT = PHASE1_READY_PATH.parent
assert len(discovered_train_leaves) == 1, discovered_train_leaves
assert len(discovered_val_leaves) == 1, discovered_val_leaves
COCO_TRAIN_LEAF = discovered_train_leaves[0]
COCO_VAL_LEAF = discovered_val_leaves[0]

blip_candidates = []
for source_manifest_path in DISCOVERED_FILE_MATCHES[
    "MODEL_SOURCE_MANIFEST.json"
]:
    try:
        payload = read_json(source_manifest_path)
    except Exception:
        continue
    if payload.get("repo_id") != EXPECTED_REPO_ID:
        continue
    if payload.get("resolved_commit") != EXPECTED_REVISION:
        continue
    blip_candidates.append(
        (source_manifest_path.parent.resolve(), source_manifest_path.resolve())
    )
assert len(blip_candidates) == 1, (
    "Cần gắn đúng một BLIP offline package có MODEL_SOURCE_MANIFEST.json "
    f"với revision {EXPECTED_REVISION}; tìm thấy {blip_candidates}"
)
BLIP_SOURCE_DIR, BLIP_SOURCE_MANIFEST_PATH = blip_candidates[0]

cell_audit_path = AUDIT_DIR / "phase3f_cell01_environment_discovery.json"
write_json(
    cell_audit_path,
    {
        "schema_version": "phase3f_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "transformers": transformers.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "gpu_memory_gib": (
            torch.cuda.get_device_properties(0).total_memory / 1024**3
        ),
        "phase3e_authorization": str(PHASE3E_AUTH_PATH),
        "phase1_ready": str(PHASE1_READY_PATH),
        "coco_train_leaf": str(COCO_TRAIN_LEAF),
        "coco_val_leaf": str(COCO_VAL_LEAF),
        "blip_source": str(BLIP_SOURCE_DIR),
        "repo_id": EXPECTED_REPO_ID,
        "resolved_revision": EXPECTED_REVISION,
        "offline_mode": True,
        "model_loaded": False,
        "caption_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03F CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("GPU                       :", torch.cuda.get_device_name(0))
print("Transformers              :", transformers.__version__)
print("Teacher B authorization   : FOUND")
print("Phase 01                  : FOUND")
print("MS COCO image roots       : FOUND")
print("BLIP offline source       : FOUND")
print("Repository                :", EXPECTED_REPO_ID)
print("Resolved revision         :", EXPECTED_REVISION)
print("Model loaded              : False")
print("Caption read              : False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 11 cells passed")
print("Next                      : verify Phase 03E and Phase 01")


PHASE 03F CELL 1 STATUS
Status                    : PASSED
GPU                       : Tesla T4
Transformers              : 5.0.0
Teacher B authorization   : FOUND
Phase 01                  : FOUND
MS COCO image roots       : FOUND
BLIP offline source       : FOUND
Repository                : Salesforce/blip-image-captioning-base
Resolved revision         : 82a37760796d32b1411fe092ab5d4e227313294b
Model loaded              : False
Caption read              : False
Test Set status           : SEALED
Progress                  : 1 / 11 cells passed
Next                      : verify Phase 03E and Phase 01


## Cell 2 — Xác minh Phase 03E và Phase 01

### Công việc

- kiểm tra Teacher B authorization và Teacher A dependency;
- xác minh 12 manifest/checksum của Phase 03E;
- xác minh các development artifact cần dùng từ Phase 01;
- không mở hoặc hash Test Parquet.

### Thời gian dự kiến

10–60 giây.

### Tiến độ hiện tại

Cell 2/11 — xác minh upstream contract và leakage boundary.

### Đầu ra dự kiến

```text
Status                    : PASSED
Phase 03E manifest        : 12 / 12 VERIFIED
Phase 01 artifacts        : 7 / 7 VERIFIED
Test Parquet opened       : False
Progress                  : 2 / 11 cells passed
```


In [2]:
import csv

CELL_STARTED_AT = time.perf_counter()
required_from_cell1 = [
    "PHASE3E_AUTH_PATH",
    "PHASE3E_ROOT",
    "PHASE1_READY_PATH",
    "PHASE1_ROOT",
    "BLIP_SOURCE_DIR",
]
missing = [name for name in required_from_cell1 if name not in globals()]
assert not missing, f"Thiếu biến Cell 1: {missing}"

phase3e_auth = read_json(PHASE3E_AUTH_PATH)
assert phase3e_auth["status"] == "ready"
assert phase3e_auth["teacher_id"] == "B"
assert phase3e_auth["candidate_model_identifier"] == EXPECTED_REPO_ID
assert phase3e_auth["model_loaded"] is False
assert phase3e_auth["training_started"] is False
assert phase3e_auth["test_set_status"] == "sealed"
assert phase3e_auth["test_set_used"] is False

phase3e_complete_path = PHASE3E_ROOT / phase3e_auth[
    "teacher_a_three_seed_complete"
]["relative_path"]
assert sha256_file(phase3e_complete_path) == phase3e_auth[
    "teacher_a_three_seed_complete"
]["sha256"]
phase3e_complete = read_json(phase3e_complete_path)
assert phase3e_complete["status"] == "complete"
assert phase3e_complete["qualified_for_kd"] is True
assert phase3e_complete["stable_across_seeds"] is True
assert phase3e_complete["test_set_used"] is False

phase3e_manifest_path = (
    PHASE3E_ROOT / "manifests" / "phase3e_manifest.csv"
)
with open(phase3e_manifest_path, newline="", encoding="utf-8") as source:
    phase3e_rows = list(csv.DictReader(source))
assert len(phase3e_rows) == 12
phase3e_failures = []
for row in phase3e_rows:
    candidate = (PHASE3E_ROOT / row["relative_path"]).resolve()
    assert candidate.is_relative_to(PHASE3E_ROOT.resolve())
    if not candidate.is_file():
        phase3e_failures.append(
            {"relative_path": row["relative_path"], "reason": "missing"}
        )
        continue
    if candidate.stat().st_size != int(row["size_bytes"]):
        phase3e_failures.append(
            {"relative_path": row["relative_path"], "reason": "size"}
        )
    if sha256_file(candidate) != row["sha256"]:
        phase3e_failures.append(
            {"relative_path": row["relative_path"], "reason": "sha256"}
        )
assert not phase3e_failures, phase3e_failures

phase1_ready = read_json(PHASE1_READY_PATH)
assert phase1_ready["status"] == "ready"
assert phase1_ready["test_set_status"] == "sealed"
assert phase1_ready["test_used"] is False
PHASE1_SHA256SUMS_PATH = exactly_one(PHASE1_ROOT, "SHA256SUMS.txt")
assert sha256_file(PHASE1_SHA256SUMS_PATH) == phase1_ready[
    "sha256sums_sha256"
]
phase1_sums = {}
for line in PHASE1_SHA256SUMS_PATH.read_text(encoding="utf-8").splitlines():
    if "  " not in line:
        continue
    digest, relative = line.split("  ", 1)
    phase1_sums[relative.strip().replace("\\", "/")] = digest.strip()

phase1_required_names = [
    "mscoco_train.parquet",
    "mscoco_validation.parquet",
    "mscoco_image_manifest.parquet",
    "experiment_contract_v2.json",
    "word_vocabulary_minfreq5.json",
    "tokenizer_contract.json",
    "sequence_encoding_contract.json",
]
PHASE1_ARTIFACTS = {}
for name in phase1_required_names:
    path = exactly_one(PHASE1_ROOT, name)
    matches = [
        (relative, digest)
        for relative, digest in phase1_sums.items()
        if relative.endswith("/" + name) or relative == name
    ]
    assert len(matches) == 1, f"Không tìm thấy checksum duy nhất cho {name}"
    assert sha256_file(path) == matches[0][1]
    PHASE1_ARTIFACTS[name] = path

assert not any(
    path.name == "mscoco_test.parquet"
    for path in PHASE1_ARTIFACTS.values()
)

cell_audit_path = (
    AUDIT_DIR / "phase3f_cell02_upstream_contract_integrity.json"
)
write_json(
    cell_audit_path,
    {
        "schema_version": "phase3f_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "phase3e_manifest_rows": len(phase3e_rows),
        "phase3e_manifest_failures": phase3e_failures,
        "teacher_a_qualified_for_kd": True,
        "phase1_verified_artifacts": {
            name: {
                "path": str(path),
                "sha256": sha256_file(path),
            }
            for name, path in PHASE1_ARTIFACTS.items()
        },
        "test_parquet_opened": False,
        "test_parquet_hashed": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03F CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Phase 03E authorization   : VERIFIED")
print("Teacher A dependency      : VERIFIED")
print("Phase 03E manifest        : 12 / 12 VERIFIED")
print("Phase 01 artifacts        : 7 / 7 VERIFIED")
print("Test Parquet opened       : False")
print("Test Parquet hashed       : False")
print("Test Set status           : SEALED")
print("Progress                  : 2 / 11 cells passed")
print("Next                      : verify exact BLIP source")


PHASE 03F CELL 2 STATUS
Status                    : PASSED
Phase 03E authorization   : VERIFIED
Teacher A dependency      : VERIFIED
Phase 03E manifest        : 12 / 12 VERIFIED
Phase 01 artifacts        : 7 / 7 VERIFIED
Test Parquet opened       : False
Test Parquet hashed       : False
Test Set status           : SEALED
Progress                  : 2 / 11 cells passed
Next                      : verify exact BLIP source


## Cell 3 — Khóa exact BLIP offline source

### Công việc

- xác minh repo và full commit hash;
- kiểm tra kích thước/SHA-256 từng file upstream;
- xác minh config, processor, tokenizer và PyTorch weight;
- ghi source contract; tuyệt đối không download mạng.

### Thời gian dự kiến

1–4 phút do phải hash khoảng 1 GB model weights.

### Tiến độ hiện tại

Cell 3/11 — khóa nguồn pretrained bất biến.

### Đầu ra dự kiến

```text
Status                    : PASSED
Repository                : Salesforce/blip-image-captioning-base
Resolved revision         : 82a37760796d32b1411fe092ab5d4e227313294b
Weight files              : 1
Network download          : False
Progress                  : 3 / 11 cells passed
```


In [3]:
CELL_STARTED_AT = time.perf_counter()
required_from_cell2 = ["PHASE1_ARTIFACTS", "phase3e_auth"]
missing = [name for name in required_from_cell2 if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–2: {missing}"

BLIP_SOURCE_MANIFEST = read_json(BLIP_SOURCE_MANIFEST_PATH)
assert BLIP_SOURCE_MANIFEST["repo_id"] == EXPECTED_REPO_ID
assert BLIP_SOURCE_MANIFEST["requested_revision"] == EXPECTED_REVISION
assert BLIP_SOURCE_MANIFEST["resolved_commit"] == EXPECTED_REVISION
assert BLIP_SOURCE_MANIFEST["framework"] == "pytorch"

source_failures = []
source_files = []
for item in BLIP_SOURCE_MANIFEST["files"]:
    relative = item["relative_path"]
    path = (BLIP_SOURCE_DIR / relative).resolve()
    assert path.is_relative_to(BLIP_SOURCE_DIR.resolve())
    if not path.is_file():
        source_failures.append({"file": relative, "reason": "missing"})
        continue
    if path.stat().st_size != int(item["size_bytes"]):
        source_failures.append({"file": relative, "reason": "size"})
    actual_sha = sha256_file(path)
    if actual_sha != item["sha256"]:
        source_failures.append({"file": relative, "reason": "sha256"})
    source_files.append(path)
assert not source_failures, source_failures

source_names = {path.name for path in source_files}
required_names = {
    "config.json",
    "preprocessor_config.json",
    "tokenizer_config.json",
    "vocab.txt",
}
assert required_names.issubset(source_names)
BLIP_WEIGHT_FILES = sorted(
    path
    for path in source_files
    if path.name == "pytorch_model.bin"
    or path.suffix == ".safetensors"
)
assert len(BLIP_WEIGHT_FILES) == 1, BLIP_WEIGHT_FILES
if BLIP_WEIGHT_FILES[0].suffix == ".bin":
    assert zipfile.is_zipfile(BLIP_WEIGHT_FILES[0])

BLIP_CONFIG_PATH = next(
    path for path in source_files if path.name == "config.json"
)
BLIP_PREPROCESSOR_CONFIG_PATH = next(
    path
    for path in source_files
    if path.name == "preprocessor_config.json"
)
BLIP_TOKENIZER_CONFIG_PATH = next(
    path
    for path in source_files
    if path.name == "tokenizer_config.json"
)
BLIP_CONFIG_RAW = read_json(BLIP_CONFIG_PATH)
BLIP_PREPROCESSOR_RAW = read_json(BLIP_PREPROCESSOR_CONFIG_PATH)
BLIP_TOKENIZER_RAW = read_json(BLIP_TOKENIZER_CONFIG_PATH)
assert BLIP_CONFIG_RAW["model_type"] == "blip"
assert "BlipForConditionalGeneration" in BLIP_CONFIG_RAW["architectures"]
assert BLIP_PREPROCESSOR_RAW["image_processor_type"] == "BlipImageProcessor"
assert BLIP_TOKENIZER_RAW["tokenizer_class"] == "BertTokenizer"

BLIP_SOURCE_CONTRACT_PATH = (
    CONFIG_DIR / "teacher_b_blip_offline_source_contract.json"
)
write_json(
    BLIP_SOURCE_CONTRACT_PATH,
    {
        "contract_version": "phase3_teacher_b_blip_source_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "repo_id": EXPECTED_REPO_ID,
        "requested_revision": EXPECTED_REVISION,
        "resolved_commit": EXPECTED_REVISION,
        "license": BLIP_SOURCE_MANIFEST.get("license"),
        "source_manifest": {
            "path": str(BLIP_SOURCE_MANIFEST_PATH),
            "sha256": sha256_file(BLIP_SOURCE_MANIFEST_PATH),
        },
        "files": [
            {
                "relative_path": path.relative_to(
                    BLIP_SOURCE_DIR
                ).as_posix(),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
            for path in source_files
        ],
        "weight_files": [
            {
                "relative_path": path.relative_to(
                    BLIP_SOURCE_DIR
                ).as_posix(),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
            for path in BLIP_WEIGHT_FILES
        ],
        "offline_only": True,
        "network_download": False,
        "model_loaded": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

cell_audit_path = AUDIT_DIR / "phase3f_cell03_blip_source_integrity.json"
write_json(
    cell_audit_path,
    {
        "schema_version": "phase3f_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "source_file_count": len(source_files),
        "source_failures": source_failures,
        "source_bytes": sum(path.stat().st_size for path in source_files),
        "weight_file_count": len(BLIP_WEIGHT_FILES),
        "weight_container_valid": True,
        "source_contract": str(BLIP_SOURCE_CONTRACT_PATH),
        "network_download": False,
        "model_loaded": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03F CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Repository                :", EXPECTED_REPO_ID)
print("Resolved revision         :", EXPECTED_REVISION)
print("Source files              :", len(source_files))
print("Source failures           : 0")
print("Weight files              :", len(BLIP_WEIGHT_FILES))
print("Weight SHA-256            :", sha256_file(BLIP_WEIGHT_FILES[0]))
print("Network download          : False")
print("Model loaded              : False")
print("Test Set status           : SEALED")
print("Progress                  : 3 / 11 cells passed")
print("Next                      : lock processor and tokenizer")


PHASE 03F CELL 3 STATUS
Status                    : PASSED
Repository                : Salesforce/blip-image-captioning-base
Resolved revision         : 82a37760796d32b1411fe092ab5d4e227313294b
Source files              : 19
Source failures           : 0
Weight files              : 1
Weight SHA-256            : d6638651a5526cc2ede56f2b5104d6851b0755816d220e5e046870430180c767
Network download          : False
Model loaded              : False
Test Set status           : SEALED
Progress                  : 3 / 11 cells passed
Next                      : lock processor and tokenizer


## Cell 4 — BLIP image processor và native tokenizer

### Công việc

- load processor hoàn toàn từ offline package;
- khóa ảnh đầu vào 384×384, mean/std và interpolation;
- khóa native WordPiece tokenizer và special token IDs;
- chứng minh BLIP vocabulary khác shared Student vocabulary;
- tiếp tục cấm logit KD trực tiếp.

### Thời gian dự kiến

5–20 giây.

### Tiến độ hiện tại

Cell 4/11 — khóa hai miền đầu vào của Teacher B.

### Đầu ra dự kiến

```text
Status                    : PASSED
Image size                : 384 x 384
Tokenizer                 : BERT WordPiece
Vocabulary aligned        : False
Direct logit KD           : PROHIBITED
Progress                  : 4 / 11 cells passed
```


In [4]:
CELL_STARTED_AT = time.perf_counter()
required_from_cell3 = [
    "BLIP_SOURCE_CONTRACT_PATH",
    "BLIP_CONFIG_RAW",
    "BLIP_PREPROCESSOR_RAW",
    "BLIP_TOKENIZER_RAW",
]
missing = [name for name in required_from_cell3 if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–3: {missing}"

from transformers import BlipProcessor

BLIP_PROCESSOR = BlipProcessor.from_pretrained(
    str(BLIP_SOURCE_DIR),
    local_files_only=True,
    use_fast=False,
)
BLIP_TOKENIZER = BLIP_PROCESSOR.tokenizer
BLIP_IMAGE_PROCESSOR = BLIP_PROCESSOR.image_processor

# Transformers 5 no longer injects the two BLIP-specific tokens that the
# upstream 4.x checkpoint expects. Add them idempotently and lock their IDs.
tokenizer_vocabulary = BLIP_TOKENIZER.get_vocab()
if "[DEC]" not in tokenizer_vocabulary:
    added_dec = BLIP_TOKENIZER.add_special_tokens(
        {"bos_token": "[DEC]"}
    )
    assert added_dec == 1
else:
    BLIP_TOKENIZER.bos_token = "[DEC]"
tokenizer_vocabulary = BLIP_TOKENIZER.get_vocab()
if "[ENC]" not in tokenizer_vocabulary:
    current_additional = BLIP_TOKENIZER.special_tokens_map.get(
        "additional_special_tokens", []
    )
    if isinstance(current_additional, str):
        current_additional = [current_additional]
    added_enc = BLIP_TOKENIZER.add_special_tokens(
        {
            "additional_special_tokens": list(current_additional)
            + ["[ENC]"]
        }
    )
    assert added_enc == 1
assert BLIP_TOKENIZER.convert_tokens_to_ids("[DEC]") == 30522
assert BLIP_TOKENIZER.convert_tokens_to_ids("[ENC]") == 30523
assert len(BLIP_TOKENIZER) == 30524

size_value = BLIP_IMAGE_PROCESSOR.size
if isinstance(size_value, dict):
    if "height" in size_value:
        image_height = int(size_value["height"])
        image_width = int(size_value["width"])
    else:
        image_height = int(size_value.get("shortest_edge", 384))
        image_width = image_height
else:
    image_height = image_width = int(size_value)
assert (image_height, image_width) == (384, 384)

expected_mean = [0.48145466, 0.4578275, 0.40821073]
expected_std = [0.26862954, 0.26130258, 0.27577711]
actual_mean = [float(value) for value in BLIP_IMAGE_PROCESSOR.image_mean]
actual_std = [float(value) for value in BLIP_IMAGE_PROCESSOR.image_std]
assert np.allclose(actual_mean, expected_mean, rtol=0, atol=1e-8)
assert np.allclose(actual_std, expected_std, rtol=0, atol=1e-8)

additional_special_tokens = BLIP_TOKENIZER.special_tokens_map.get(
    "additional_special_tokens", []
)
if isinstance(additional_special_tokens, str):
    additional_special_tokens = [additional_special_tokens]
# Transformers 5 may expose AddedToken objects here. Resolve the locked BLIP
# token by its canonical string instead of depending on that representation.
additional_special_tokens = [
    str(token) for token in additional_special_tokens
]
if "[ENC]" not in additional_special_tokens:
    additional_special_tokens.append("[ENC]")
additional_special_token_ids = [
    int(BLIP_TOKENIZER.convert_tokens_to_ids("[ENC]"))
]

special_ids = {
    "pad_token_id": BLIP_TOKENIZER.pad_token_id,
    "unk_token_id": BLIP_TOKENIZER.unk_token_id,
    "cls_token_id": BLIP_TOKENIZER.cls_token_id,
    "sep_token_id": BLIP_TOKENIZER.sep_token_id,
    "bos_token_id": BLIP_TOKENIZER.bos_token_id,
    "eos_token_id": BLIP_TOKENIZER.eos_token_id,
    "additional_special_tokens_ids": additional_special_token_ids,
}
assert special_ids["pad_token_id"] is not None
assert special_ids["unk_token_id"] is not None
assert special_ids["cls_token_id"] is not None
assert special_ids["sep_token_id"] is not None
assert special_ids["bos_token_id"] == 30522
assert special_ids["additional_special_tokens_ids"] == [30523]
assert len(BLIP_TOKENIZER) >= BLIP_TOKENIZER.vocab_size

student_vocabulary = read_json(
    PHASE1_ARTIFACTS["word_vocabulary_minfreq5.json"]
)
student_vocab_size = int(student_vocabulary["vocabulary_size"])
blip_vocab_size = len(BLIP_TOKENIZER)
VOCABULARY_ALIGNED = (
    student_vocab_size == blip_vocab_size
    and student_vocabulary["special_token_ids"]["<pad>"]
    == BLIP_TOKENIZER.pad_token_id
)
assert VOCABULARY_ALIGNED is False

BLIP_PROCESSOR_CONTRACT_PATH = (
    CONFIG_DIR / "teacher_b_blip_processor_tokenizer_contract.json"
)
write_json(
    BLIP_PROCESSOR_CONTRACT_PATH,
    {
        "contract_version": "phase3_teacher_b_processor_tokenizer_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "image_processor": {
            "class": type(BLIP_IMAGE_PROCESSOR).__name__,
            "use_fast": False,
            "height": image_height,
            "width": image_width,
            "do_resize": bool(BLIP_IMAGE_PROCESSOR.do_resize),
            "do_normalize": bool(BLIP_IMAGE_PROCESSOR.do_normalize),
            "mean": actual_mean,
            "std": actual_std,
            "resample": str(BLIP_IMAGE_PROCESSOR.resample),
        },
        "tokenizer": {
            "class": type(BLIP_TOKENIZER).__name__,
            "algorithm": "BERT WordPiece",
            "do_lower_case": bool(
                getattr(BLIP_TOKENIZER, "do_lower_case", True)
            ),
            "base_vocab_size": int(BLIP_TOKENIZER.vocab_size),
            "effective_vocab_size": int(blip_vocab_size),
            "model_max_length": int(BLIP_TOKENIZER.model_max_length),
            "special_token_ids": special_ids,
        },
        "student_shared_vocabulary_size": student_vocab_size,
        "vocabulary_aligned": False,
        "direct_logit_kd": "prohibited",
        "feature_kd": "allowed_after_projection_contract",
        "sequence_kd": "allowed_after_generation_contract",
        "offline_only": True,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

cell_audit_path = (
    AUDIT_DIR / "phase3f_cell04_processor_tokenizer.json"
)
write_json(
    cell_audit_path,
    {
        "schema_version": "phase3f_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "image_shape": [3, image_height, image_width],
        "image_mean": actual_mean,
        "image_std": actual_std,
        "tokenizer_class": type(BLIP_TOKENIZER).__name__,
        "blip_vocabulary_size": blip_vocab_size,
        "student_vocabulary_size": student_vocab_size,
        "special_token_ids": special_ids,
        "vocabulary_aligned": False,
        "direct_logit_kd": "prohibited",
        "processor_contract": str(BLIP_PROCESSOR_CONTRACT_PATH),
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03F CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Image processor           :", type(BLIP_IMAGE_PROCESSOR).__name__)
print("Image size                : 384 x 384")
print("Tokenizer                 :", type(BLIP_TOKENIZER).__name__)
print("Tokenizer vocabulary      :", blip_vocab_size)
print("Student vocabulary        :", student_vocab_size)
print("Vocabulary aligned        : False")
print("Direct logit KD           : PROHIBITED")
print("Feature/sequence KD       : CANDIDATE")
print("Test Set status           : SEALED")
print("Progress                  : 4 / 11 cells passed")
print("Next                      : audit native token length and data")


PHASE 03F CELL 4 STATUS
Status                    : PASSED
Image processor           : BlipImageProcessor
Image size                : 384 x 384
Tokenizer                 : BertTokenizer
Tokenizer vocabulary      : 30524
Student vocabulary        : 10080
Vocabulary aligned        : False
Direct logit KD           : PROHIBITED
Feature/sequence KD       : CANDIDATE
Test Set status           : SEALED
Progress                  : 4 / 11 cells passed
Next                      : audit native token length and data


## Cell 5 — Development data và native token-length policy

### Công việc

- mở Train/Validation Parquet, không mở Test Parquet;
- dựng path map cho 113.287 Train và 5.000 Validation image;
- tokenize toàn bộ caption bằng BLIP WordPiece;
- đo sensitivity tại 24/32/40/48/64 token;
- khóa `max_text_length=40` nếu đạt truncation/retention gate.

### Thời gian dự kiến

2–8 phút, tùy CPU.

### Tiến độ hiện tại

Cell 5/11 — khóa dữ liệu phát triển và chiều dài chuỗi.

### Đầu ra dự kiến

```text
Status                    : PASSED
Training captions         : 566747
Validation captions       : 25010
Maximum text length       : 40
Test Parquet opened       : False
Progress                  : 5 / 11 cells passed
```


In [5]:
CELL_STARTED_AT = time.perf_counter()
required_from_cell4 = [
    "BLIP_PROCESSOR",
    "BLIP_TOKENIZER",
    "BLIP_PROCESSOR_CONTRACT_PATH",
]
missing = [name for name in required_from_cell4 if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–4: {missing}"

import pyarrow.parquet as pq

TRAIN_PARQUET = PHASE1_ARTIFACTS["mscoco_train.parquet"]
VALIDATION_PARQUET = PHASE1_ARTIFACTS["mscoco_validation.parquet"]
IMAGE_MANIFEST_PARQUET = PHASE1_ARTIFACTS[
    "mscoco_image_manifest.parquet"
]

train_pf = pq.ParquetFile(TRAIN_PARQUET)
validation_pf = pq.ParquetFile(VALIDATION_PARQUET)
manifest_pf = pq.ParquetFile(IMAGE_MANIFEST_PARQUET)
expected_caption_columns = {
    "annotation_id",
    "image_id",
    "karpathy_split",
    "source_partition",
    "caption",
}
assert expected_caption_columns.issubset(
    set(train_pf.schema_arrow.names)
)
assert expected_caption_columns.issubset(
    set(validation_pf.schema_arrow.names)
)
assert {
    "image_id",
    "experiment_split",
    "source_partition",
    "image_path",
}.issubset(set(manifest_pf.schema_arrow.names))
assert train_pf.metadata.num_rows == 566747
assert validation_pf.metadata.num_rows == 25010
assert manifest_pf.metadata.num_rows == 123287

train_rows = pq.read_table(
    TRAIN_PARQUET,
    columns=[
        "annotation_id",
        "image_id",
        "karpathy_split",
        "source_partition",
        "caption",
    ],
).to_pylist()
validation_rows = pq.read_table(
    VALIDATION_PARQUET,
    columns=[
        "annotation_id",
        "image_id",
        "karpathy_split",
        "source_partition",
        "caption",
    ],
).to_pylist()
manifest_rows = pq.read_table(
    IMAGE_MANIFEST_PARQUET,
    columns=[
        "image_id",
        "experiment_split",
        "source_partition",
        "image_path",
    ],
).to_pylist()


def current_image_path(row):
    file_name = Path(row["image_path"]).name
    if row["source_partition"] == "train2014":
        return COCO_TRAIN_LEAF / file_name
    if row["source_partition"] == "val2014":
        return COCO_VAL_LEAF / file_name
    raise AssertionError(row)


TRAIN_IMAGE_PATHS = {}
VALIDATION_IMAGE_PATHS = {}
HELD_OUT_TEST_IDS = set()
for row in manifest_rows:
    image_id = int(row["image_id"])
    split = row["experiment_split"]
    if split == "train":
        TRAIN_IMAGE_PATHS[image_id] = current_image_path(row)
    elif split == "validation":
        VALIDATION_IMAGE_PATHS[image_id] = current_image_path(row)
    elif split == "held_out_test":
        HELD_OUT_TEST_IDS.add(image_id)
    else:
        raise AssertionError(f"Unknown experiment split: {split}")
assert len(TRAIN_IMAGE_PATHS) == 113287
assert len(VALIDATION_IMAGE_PATHS) == 5000
assert len(HELD_OUT_TEST_IDS) == 5000
assert not (set(TRAIN_IMAGE_PATHS) & set(VALIDATION_IMAGE_PATHS))
assert not (set(TRAIN_IMAGE_PATHS) & HELD_OUT_TEST_IDS)
assert not (set(VALIDATION_IMAGE_PATHS) & HELD_OUT_TEST_IDS)
path_probe = (
    list(TRAIN_IMAGE_PATHS.values())[:128]
    + list(VALIDATION_IMAGE_PATHS.values())[:128]
)
missing_paths = [str(path) for path in path_probe if not path.is_file()]
assert not missing_paths, missing_paths

train_captions = [row["caption"] for row in train_rows]
validation_captions = [row["caption"] for row in validation_rows]
assert all(isinstance(text, str) and text.strip() for text in train_captions)
assert all(
    isinstance(text, str) and text.strip()
    for text in validation_captions
)


def native_lengths(captions, batch_size=8192):
    result = []
    for start in range(0, len(captions), batch_size):
        encoded = BLIP_TOKENIZER(
            captions[start : start + batch_size],
            add_special_tokens=True,
            truncation=False,
            padding=False,
        )
        result.extend(len(ids) for ids in encoded["input_ids"])
    return result


train_native_lengths = native_lengths(train_captions)
validation_native_lengths = native_lengths(validation_captions)
assert len(train_native_lengths) == 566747
assert len(validation_native_lengths) == 25010


def length_summary(values):
    ordered = sorted(values)
    def percentile(q):
        return ordered[int(round(q * (len(ordered) - 1)))]
    return {
        "minimum": ordered[0],
        "p50": percentile(0.50),
        "p90": percentile(0.90),
        "p95": percentile(0.95),
        "p99": percentile(0.99),
        "maximum": ordered[-1],
        "mean": statistics.fmean(ordered),
    }


def sensitivity(values, limit):
    truncated = sum(length > limit for length in values)
    retained = sum(min(length, limit) for length in values)
    total = sum(values)
    return {
        "maximum_full_tokens": limit,
        "truncated_captions": truncated,
        "truncation_rate": truncated / len(values),
        "token_retention": retained / total,
    }


length_candidates = [24, 32, 40, 48, 64]
train_sensitivity = {
    str(limit): sensitivity(train_native_lengths, limit)
    for limit in length_candidates
}
validation_sensitivity = {
    str(limit): sensitivity(validation_native_lengths, limit)
    for limit in length_candidates
}
MAX_TEXT_LENGTH = 40
train_selected = train_sensitivity[str(MAX_TEXT_LENGTH)]
validation_selected = validation_sensitivity[str(MAX_TEXT_LENGTH)]
assert train_selected["truncation_rate"] <= 0.001
assert train_selected["token_retention"] >= 0.9995
assert validation_selected["truncation_rate"] <= 0.001
assert validation_selected["token_retention"] >= 0.9995


def first_unique_samples(rows, path_map, count):
    selected = []
    seen = set()
    for row in rows:
        image_id = int(row["image_id"])
        if image_id in seen:
            continue
        assert image_id in path_map
        selected.append(
            {
                "annotation_id": int(row["annotation_id"]),
                "image_id": image_id,
                "caption": row["caption"],
                "image_path": path_map[image_id],
            }
        )
        seen.add(image_id)
        if len(selected) == count:
            return selected
    raise AssertionError("Không đủ unique image samples.")


TRAIN_SMOKE_SAMPLES = first_unique_samples(
    train_rows, TRAIN_IMAGE_PATHS, 4
)
VALIDATION_SMOKE_SAMPLES = first_unique_samples(
    validation_rows, VALIDATION_IMAGE_PATHS, 4
)
assert not (
    {item["image_id"] for item in TRAIN_SMOKE_SAMPLES}
    & {item["image_id"] for item in VALIDATION_SMOKE_SAMPLES}
)

BLIP_SEQUENCE_CONTRACT_PATH = (
    CONFIG_DIR / "teacher_b_blip_native_sequence_contract.json"
)
write_json(
    BLIP_SEQUENCE_CONTRACT_PATH,
    {
        "contract_version": "phase3_teacher_b_native_sequence_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "tokenizer": type(BLIP_TOKENIZER).__name__,
        "vocabulary_source": "BLIP offline package only",
        "maximum_full_tokens": MAX_TEXT_LENGTH,
        "padding": "dynamic per batch",
        "truncation": "right",
        "train_length_summary": length_summary(train_native_lengths),
        "validation_length_summary": length_summary(
            validation_native_lengths
        ),
        "train_sensitivity": train_sensitivity,
        "validation_sensitivity": validation_sensitivity,
        "selection_gate": {
            "maximum_truncation_rate": 0.001,
            "minimum_token_retention": 0.9995,
            "passed": True,
        },
        "training_captions": len(train_rows),
        "validation_captions": len(validation_rows),
        "test_parquet_opened": False,
        "test_caption_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

cell_audit_path = (
    AUDIT_DIR / "phase3f_cell05_development_data_token_length.json"
)
write_json(
    cell_audit_path,
    {
        "schema_version": "phase3f_cell05_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "training_rows": len(train_rows),
        "training_images": len(TRAIN_IMAGE_PATHS),
        "validation_rows": len(validation_rows),
        "validation_images": len(VALIDATION_IMAGE_PATHS),
        "held_out_test_ids_metadata_only": len(HELD_OUT_TEST_IDS),
        "path_probe_count": len(path_probe),
        "missing_paths": missing_paths,
        "maximum_text_length": MAX_TEXT_LENGTH,
        "train_selected_sensitivity": train_selected,
        "validation_selected_sensitivity": validation_selected,
        "sequence_contract": str(BLIP_SEQUENCE_CONTRACT_PATH),
        "train_caption_read": True,
        "validation_caption_read": True,
        "test_parquet_opened": False,
        "test_caption_read": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03F CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Training captions         :", len(train_rows))
print("Training images           :", len(TRAIN_IMAGE_PATHS))
print("Validation captions       :", len(validation_rows))
print("Validation images         :", len(VALIDATION_IMAGE_PATHS))
print("Native train length       :", length_summary(train_native_lengths))
print("Native validation length  :", length_summary(validation_native_lengths))
print("Maximum text length       :", MAX_TEXT_LENGTH)
print("Train truncation rate     :", train_selected["truncation_rate"])
print("Train token retention     :", train_selected["token_retention"])
print("Validation truncation rate:", validation_selected["truncation_rate"])
print("Validation token retention:", validation_selected["token_retention"])
print("Test Parquet opened       : False")
print("Test caption read         : False")
print("Test Set status           : SEALED")
print("Progress                  : 5 / 11 cells passed")
print("Next                      : load and inspect BLIP architecture")


PHASE 03F CELL 5 STATUS
Status                    : PASSED
Training captions         : 566747
Training images           : 113287
Validation captions       : 25010
Validation images         : 5000
Native train length       : {'minimum': 9, 'p50': 13, 'p90': 17, 'p95': 19, 'p99': 23, 'maximum': 65, 'mean': 13.811909017603975}
Native validation length  : {'minimum': 9, 'p50': 13, 'p90': 17, 'p95': 19, 'p99': 23, 'maximum': 56, 'mean': 13.82391043582567}
Maximum text length       : 40
Train truncation rate     : 0.0004905186970552998
Train token retention     : 0.9997066885985924
Validation truncation rate: 0.0003198720511795282
Validation token retention: 0.9998062105190088
Test Parquet opened       : False
Test caption read         : False
Test Set status           : SEALED
Progress                  : 5 / 11 cells passed
Next                      : load and inspect BLIP architecture


## Cell 6 — Load model và audit kiến trúc

### Công việc

- load `BlipForConditionalGeneration` từ input offline;
- xác minh vision encoder, text decoder và vocabulary;
- chạy forward trên Validation sample;
- kiểm tra visual token shape và logits shape;
- ghi parameter/memory contract.

### Thời gian dự kiến

1–4 phút.

### Tiến độ hiện tại

Cell 6/11 — xác minh model thực tế và giao diện tensor.

### Đầu ra dự kiến

```text
Status                    : PASSED
Architecture              : BlipForConditionalGeneration
Vision input              : [B,3,384,384]
Vision memory             : [B,577,768]
Network download          : False
Progress                  : 6 / 11 cells passed
```


In [6]:
CELL_STARTED_AT = time.perf_counter()
required_from_cell5 = [
    "TRAIN_SMOKE_SAMPLES",
    "VALIDATION_SMOKE_SAMPLES",
    "MAX_TEXT_LENGTH",
    "BLIP_SEQUENCE_CONTRACT_PATH",
]
missing = [name for name in required_from_cell5 if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–5: {missing}"

from PIL import Image
from transformers import BlipForConditionalGeneration


def load_rgb(path):
    with Image.open(path) as image:
        return image.convert("RGB")


def load_blip_model():
    model, loading_info = BlipForConditionalGeneration.from_pretrained(
        str(BLIP_SOURCE_DIR),
        local_files_only=True,
        output_loading_info=True,
    )
    allowed_unexpected = {
        "text_decoder.bert.embeddings.position_ids"
    }
    unexpected_keys = set(loading_info.get("unexpected_keys", []))
    missing_keys = list(loading_info.get("missing_keys", []))
    mismatched_keys = list(loading_info.get("mismatched_keys", []))
    error_messages = list(loading_info.get("error_msgs", []))
    assert unexpected_keys.issubset(allowed_unexpected), unexpected_keys
    assert not missing_keys, missing_keys
    assert not mismatched_keys, mismatched_keys
    assert not error_messages, error_messages
    model._klcn_loading_info = {
        "unexpected_keys": sorted(unexpected_keys),
        "allowed_unexpected_keys": sorted(allowed_unexpected),
        "missing_keys": missing_keys,
        "mismatched_keys": mismatched_keys,
        "error_messages": error_messages,
        "trainable_parameter_coverage_preserved": True,
        "compatibility_note": (
            "Transformers 5 treats position_ids as a non-persistent "
            "buffer; this key is not a trainable parameter."
        ),
    }
    return model.to(DEVICE)


def encode_samples(samples):
    images = [load_rgb(item["image_path"]) for item in samples]
    captions = [item["caption"] for item in samples]
    encoded = BLIP_PROCESSOR(
        images=images,
        text=captions,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_TEXT_LENGTH,
    )
    return {
        key: value.to(DEVICE)
        for key, value in encoded.items()
        if isinstance(value, torch.Tensor)
    }


torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
MODEL = load_blip_model()
MODEL.eval()
MODEL_LOADING_INFO = MODEL._klcn_loading_info

assert type(MODEL).__name__ == "BlipForConditionalGeneration"
assert MODEL.config.model_type == "blip"
assert int(MODEL.config.vision_config.image_size) == 384
assert int(MODEL.config.vision_config.patch_size) == 16
assert int(MODEL.config.vision_config.hidden_size) == 768
assert int(MODEL.config.vision_config.num_hidden_layers) == 12
assert int(MODEL.config.text_config.hidden_size) == 768
assert int(MODEL.config.text_config.num_hidden_layers) == 12
assert int(MODEL.config.text_config.vocab_size) == len(BLIP_TOKENIZER)
assert MODEL_LOADING_INFO["unexpected_keys"] in [
    [],
    ["text_decoder.bert.embeddings.position_ids"],
]
assert MODEL_LOADING_INFO["trainable_parameter_coverage_preserved"] is True

VALIDATION_SMOKE_BATCH = encode_samples(
    VALIDATION_SMOKE_SAMPLES[:2]
)
with torch.inference_mode():
    vision_output = MODEL.vision_model(
        pixel_values=VALIDATION_SMOKE_BATCH["pixel_values"],
        return_dict=True,
    )
    forward_output = MODEL(
        pixel_values=VALIDATION_SMOKE_BATCH["pixel_values"],
        input_ids=VALIDATION_SMOKE_BATCH["input_ids"],
        attention_mask=VALIDATION_SMOKE_BATCH["attention_mask"],
        return_dict=True,
    )
vision_shape = list(vision_output.last_hidden_state.shape)
logits_shape = list(forward_output.logits.shape)
assert vision_shape == [2, 577, 768]
assert logits_shape[0] == 2
assert logits_shape[1] == VALIDATION_SMOKE_BATCH["input_ids"].shape[1]
assert logits_shape[2] == len(BLIP_TOKENIZER)
assert torch.isfinite(vision_output.last_hidden_state).all()
assert torch.isfinite(forward_output.logits).all()

total_parameters = sum(parameter.numel() for parameter in MODEL.parameters())
trainable_parameters = sum(
    parameter.numel()
    for parameter in MODEL.parameters()
    if parameter.requires_grad
)
vision_parameters = sum(
    parameter.numel() for parameter in MODEL.vision_model.parameters()
)
text_parameters = total_parameters - vision_parameters
peak_gpu_mib = torch.cuda.max_memory_allocated() / 1024**2

BLIP_ARCHITECTURE_CONTRACT_PATH = (
    CONFIG_DIR / "teacher_b_blip_architecture_contract.json"
)
write_json(
    BLIP_ARCHITECTURE_CONTRACT_PATH,
    {
        "contract_version": "phase3_teacher_b_blip_architecture_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "class": type(MODEL).__name__,
        "repo_id": EXPECTED_REPO_ID,
        "resolved_revision": EXPECTED_REVISION,
        "vision": {
            "image_size": 384,
            "patch_size": 16,
            "hidden_size": 768,
            "layers": 12,
            "visual_tokens": 577,
            "memory_shape": ["B", 577, 768],
        },
        "text_decoder": {
            "hidden_size": int(MODEL.config.text_config.hidden_size),
            "layers": int(MODEL.config.text_config.num_hidden_layers),
            "vocabulary_size": int(
                MODEL.config.text_config.vocab_size
            ),
            "maximum_text_length": MAX_TEXT_LENGTH,
        },
        "parameters": {
            "total": total_parameters,
            "trainable": trainable_parameters,
            "vision": vision_parameters,
            "text_and_other": text_parameters,
        },
        "transformers_5_loading_compatibility": MODEL_LOADING_INFO,
        "forward_shapes": {
            "pixel_values": list(
                VALIDATION_SMOKE_BATCH["pixel_values"].shape
            ),
            "input_ids": list(
                VALIDATION_SMOKE_BATCH["input_ids"].shape
            ),
            "vision_memory": vision_shape,
            "logits": logits_shape,
        },
        "network_download": False,
        "validation_images_read": 2,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

cell_audit_path = (
    AUDIT_DIR / "phase3f_cell06_model_architecture_forward.json"
)
write_json(
    cell_audit_path,
    {
        "schema_version": "phase3f_cell06_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "model_class": type(MODEL).__name__,
        "total_parameters": total_parameters,
        "trainable_parameters": trainable_parameters,
        "vision_shape": vision_shape,
        "logits_shape": logits_shape,
        "forward_finite": True,
        "loading_info": MODEL_LOADING_INFO,
        "peak_gpu_mib": peak_gpu_mib,
        "architecture_contract": str(
            BLIP_ARCHITECTURE_CONTRACT_PATH
        ),
        "network_download": False,
        "validation_images_read": 2,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03F CELL 6 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Architecture              :", type(MODEL).__name__)
print("Total parameters          :", total_parameters)
print("Trainable parameters      :", trainable_parameters)
print("Allowed unexpected keys   :", MODEL_LOADING_INFO["unexpected_keys"])
print("Vision input              :", list(VALIDATION_SMOKE_BATCH["pixel_values"].shape))
print("Vision memory             :", vision_shape)
print("Text logits               :", logits_shape)
print("Peak GPU memory           :", round(peak_gpu_mib, 2), "MiB")
print("Network download          : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Progress                  : 6 / 11 cells passed")
print("Next                      : run loss/backward smoke test")


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: /kaggle/input/models/nguynthnhdan/blip-image-captioning-base-offline/pytorch/82a37760-pytorch/1/salesforce_blip_image_captioning_base_82a37760
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:

PHASE 03F CELL 6 STATUS
Status                    : PASSED
Architecture              : BlipForConditionalGeneration
Total parameters          : 247444600
Trainable parameters      : 247444600
Allowed unexpected keys   : ['text_decoder.bert.embeddings.position_ids']
Vision input              : [2, 3, 384, 384]
Vision memory             : [2, 577, 768]
Text logits               : [2, 11, 30524]
Peak GPU memory           : 1056.34 MiB
Network download          : False
Test images read          : False
Test Set status           : SEALED
Progress                  : 6 / 11 cells passed
Next                      : run loss/backward smoke test


## Cell 7 — Loss và backward smoke test

### Công việc

- chạy teacher-forced cross-entropy trên Train sample;
- backward qua vision encoder và text decoder;
- kiểm tra loss, gradient và tensor hữu hạn;
- không optimizer step và không thay đổi trọng số.

### Thời gian dự kiến

15–60 giây.

### Tiến độ hiện tại

Cell 7/11 — xác minh đường truyền gradient end-to-end.

### Đầu ra dự kiến

```text
Status                    : PASSED
Cross-entropy             : FINITE
Vision gradients          : PRESENT
Text gradients            : PRESENT
Optimizer step            : False
Progress                  : 7 / 11 cells passed
```


In [7]:
CELL_STARTED_AT = time.perf_counter()
required_from_cell6 = [
    "MODEL",
    "load_blip_model",
    "encode_samples",
    "BLIP_ARCHITECTURE_CONTRACT_PATH",
]
missing = [name for name in required_from_cell6 if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–6: {missing}"

TRAIN_SMOKE_BATCH = encode_samples(TRAIN_SMOKE_SAMPLES[:2])
labels = TRAIN_SMOKE_BATCH["input_ids"].clone()
labels[labels == BLIP_TOKENIZER.pad_token_id] = -100

MODEL.train()
MODEL.zero_grad(set_to_none=True)
torch.cuda.reset_peak_memory_stats()
with torch.autocast(device_type="cuda", dtype=torch.float16):
    train_output = MODEL(
        pixel_values=TRAIN_SMOKE_BATCH["pixel_values"],
        input_ids=TRAIN_SMOKE_BATCH["input_ids"],
        attention_mask=TRAIN_SMOKE_BATCH["attention_mask"],
        labels=labels,
        return_dict=True,
    )
    smoke_loss = train_output.loss
assert torch.isfinite(smoke_loss)
smoke_loss.backward()

gradient_tensors = []
nonfinite_gradient_names = []
for name, parameter in MODEL.named_parameters():
    if parameter.grad is None:
        continue
    gradient_tensors.append(name)
    if not torch.isfinite(parameter.grad).all():
        nonfinite_gradient_names.append(name)
assert not nonfinite_gradient_names, nonfinite_gradient_names
vision_gradient_count = sum(
    parameter.grad is not None
    for parameter in MODEL.vision_model.parameters()
)
text_gradient_count = sum(
    parameter.grad is not None
    for parameter in MODEL.text_decoder.parameters()
)
assert vision_gradient_count > 0
assert text_gradient_count > 0

gradient_l2 = math.sqrt(
    sum(
        float(parameter.grad.detach().float().pow(2).sum().item())
        for parameter in MODEL.parameters()
        if parameter.grad is not None
    )
)
assert math.isfinite(gradient_l2)
peak_gpu_mib = torch.cuda.max_memory_allocated() / 1024**2
MODEL.zero_grad(set_to_none=True)
MODEL.eval()

cell_audit_path = (
    AUDIT_DIR / "phase3f_cell07_loss_backward_smoke.json"
)
write_json(
    cell_audit_path,
    {
        "schema_version": "phase3f_cell07_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "batch_size": 2,
        "loss": float(smoke_loss.detach().item()),
        "gradient_l2": gradient_l2,
        "gradient_tensor_count": len(gradient_tensors),
        "vision_gradient_count": vision_gradient_count,
        "text_gradient_count": text_gradient_count,
        "nonfinite_gradient_names": nonfinite_gradient_names,
        "peak_gpu_mib": peak_gpu_mib,
        "mixed_precision": "fp16 autocast",
        "optimizer_created": False,
        "optimizer_step_performed": False,
        "weights_changed": False,
        "train_images_read": 2,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03F CELL 7 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Cross-entropy             :", float(smoke_loss.detach().item()))
print("Gradient L2              :", gradient_l2)
print("Gradient tensors          :", len(gradient_tensors))
print("Vision gradients          :", vision_gradient_count)
print("Text gradients            :", text_gradient_count)
print("Nonfinite gradients       : 0")
print("Optimizer step            : False")
print("Weights changed           : False")
print("Test images read          : False")
print("Test Set status           : SEALED")
print("Progress                  : 7 / 11 cells passed")
print("Next                      : deterministic generation audit")


PHASE 03F CELL 7 STATUS
Status                    : PASSED
Cross-entropy             : 2.6571807861328125
Gradient L2              : 24.582464188632215
Gradient tensors          : 472
Vision gradients          : 150
Text gradients            : 322
Nonfinite gradients       : 0
Optimizer step            : False
Weights changed           : False
Test images read          : False
Test Set status           : SEALED
Progress                  : 7 / 11 cells passed
Next                      : deterministic generation audit


## Cell 8 — Deterministic Beam-3 generation và feature contract

### Công việc

- sinh caption hai lần trên cùng Validation images;
- dùng Beam-3, length penalty 0,7 và tối đa 31 token mới;
- yêu cầu token IDs và beam scores lặp lại chính xác;
- khóa vision feature `[B,577,768]` cho feature KD;
- không tính metric trên Held-out Test.

### Thời gian dự kiến

20–90 giây.

### Tiến độ hiện tại

Cell 8/11 — xác minh suy luận và kênh tri thức của Teacher B.

### Đầu ra dự kiến

```text
Status                    : PASSED
Beam size / penalty       : 3 / 0.7
Token IDs repeat exact    : True
Vision feature contract   : [B,577,768]
Sequence KD               : ALLOWED
Progress                  : 8 / 11 cells passed
```


In [8]:
CELL_STARTED_AT = time.perf_counter()
required_from_cell7 = ["MODEL", "VALIDATION_SMOKE_SAMPLES"]
missing = [name for name in required_from_cell7 if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–7: {missing}"

generation_images = [
    load_rgb(item["image_path"])
    for item in VALIDATION_SMOKE_SAMPLES[:2]
]
generation_batch = BLIP_PROCESSOR(
    images=generation_images,
    return_tensors="pt",
)
generation_pixel_values = generation_batch["pixel_values"].to(DEVICE)
GENERATION_CONFIG = {
    "num_beams": 3,
    "do_sample": False,
    "length_penalty": 0.7,
    "max_new_tokens": 31,
    "min_new_tokens": 2,
    "early_stopping": True,
    "return_dict_in_generate": True,
    "output_scores": True,
}

MODEL.eval()
with torch.inference_mode():
    generation_first = MODEL.generate(
        pixel_values=generation_pixel_values,
        **GENERATION_CONFIG,
    )
    feature_first = MODEL.vision_model(
        pixel_values=generation_pixel_values,
        return_dict=True,
    ).last_hidden_state
    generation_second = MODEL.generate(
        pixel_values=generation_pixel_values,
        **GENERATION_CONFIG,
    )
    feature_second = MODEL.vision_model(
        pixel_values=generation_pixel_values,
        return_dict=True,
    ).last_hidden_state

token_ids_repeat_exact = torch.equal(
    generation_first.sequences,
    generation_second.sequences,
)
scores_first = getattr(generation_first, "sequences_scores", None)
scores_second = getattr(generation_second, "sequences_scores", None)
if scores_first is None:
    beam_scores_repeat_exact = True
    sequence_scores = [None] * len(generation_first.sequences)
else:
    beam_scores_repeat_exact = torch.equal(
        scores_first, scores_second
    )
    sequence_scores = [
        float(value)
        for value in scores_first.detach().cpu().tolist()
    ]
feature_repeat_error = float(
    (feature_first - feature_second).abs().max().item()
)
assert token_ids_repeat_exact
assert beam_scores_repeat_exact
assert feature_repeat_error == 0.0
assert list(feature_first.shape) == [2, 577, 768]

generated_captions = BLIP_PROCESSOR.batch_decode(
    generation_first.sequences,
    skip_special_tokens=True,
)
generated_captions = [text.strip() for text in generated_captions]
assert all(generated_captions)
generation_rows = []
for sample, token_ids, score, caption in zip(
    VALIDATION_SMOKE_SAMPLES[:2],
    generation_first.sequences.detach().cpu().tolist(),
    sequence_scores,
    generated_captions,
):
    generation_rows.append(
        {
            "image_id": sample["image_id"],
            "token_ids": token_ids,
            "sequence_score": score,
            "caption": caption,
        }
    )

BLIP_GENERATION_FEATURE_CONTRACT_PATH = (
    CONFIG_DIR / "teacher_b_blip_generation_feature_contract.json"
)
write_json(
    BLIP_GENERATION_FEATURE_CONTRACT_PATH,
    {
        "contract_version": "phase3_teacher_b_generation_feature_v1",
        "created_at_utc": now_utc(),
        "status": "locked",
        "generation": GENERATION_CONFIG,
        "token_ids_repeat_exact": token_ids_repeat_exact,
        "beam_scores_repeat_exact": beam_scores_repeat_exact,
        "generated_samples": generation_rows,
        "feature_kd_source": {
            "module": "vision_model final hidden state",
            "shape": ["B", 577, 768],
            "includes_cls_token": True,
            "normalization": (
                "lock projection and normalization in KD phase"
            ),
            "allowed": True,
        },
        "sequence_kd": {
            "source": "native BLIP decoded caption and sequence score",
            "allowed": True,
        },
        "direct_logit_kd": {
            "allowed": False,
            "reason": "native BLIP vocabulary is not Student vocabulary",
        },
        "validation_images_read": 2,
        "test_images_read": False,
        "test_metrics_computed": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

cell_audit_path = (
    AUDIT_DIR / "phase3f_cell08_generation_determinism_feature.json"
)
write_json(
    cell_audit_path,
    {
        "schema_version": "phase3f_cell08_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "beam_size": 3,
        "length_penalty": 0.7,
        "maximum_new_tokens": 31,
        "token_ids_repeat_exact": token_ids_repeat_exact,
        "beam_scores_repeat_exact": beam_scores_repeat_exact,
        "feature_shape": list(feature_first.shape),
        "feature_repeat_error": feature_repeat_error,
        "generated_samples": generation_rows,
        "generation_feature_contract": str(
            BLIP_GENERATION_FEATURE_CONTRACT_PATH
        ),
        "validation_images_read": 2,
        "test_images_read": False,
        "test_metrics_computed": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03F CELL 8 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Beam size / penalty       : 3 / 0.7")
print("Maximum new tokens        : 31")
print("Token IDs repeat exact    :", token_ids_repeat_exact)
print("Beam scores repeat exact  :", beam_scores_repeat_exact)
print("Feature repeat error      :", feature_repeat_error)
print("Vision feature contract   : [B,577,768]")
for row in generation_rows:
    print(f"image_id={row['image_id']} | caption={row['caption']}")
print("Feature KD                : ALLOWED WITH PROJECTION")
print("Sequence KD               : ALLOWED")
print("Direct logit KD           : PROHIBITED")
print("Test metrics computed     : False")
print("Test Set status           : SEALED")
print("Progress                  : 8 / 11 cells passed")
print("Next                      : one-batch overfit gate")


PHASE 03F CELL 8 STATUS
Status                    : PASSED
Beam size / penalty       : 3 / 0.7
Maximum new tokens        : 31
Token IDs repeat exact    : True
Beam scores repeat exact  : True
Feature repeat error      : 0.0
Vision feature contract   : [B,577,768]
image_id=106140 | caption=the plane is white and orange
image_id=107990 | caption=a boy sitting on a motorcycle
Feature KD                : ALLOWED WITH PROJECTION
Sequence KD               : ALLOWED
Direct logit KD           : PROHIBITED
Test metrics computed     : False
Test Set status           : SEALED
Progress                  : 8 / 11 cells passed
Next                      : one-batch overfit gate


## Cell 9 — One-batch overfit và FP16 gate

### Công việc

- overfit hai Train images với augmentation tắt;
- dùng FP16 + GradScaler và overflow retry tối đa 10;
- yêu cầu token accuracy ≥ 0,90 và loss ratio ≤ 0,25;
- xóa trọng số diagnostic rồi reload pristine source;
- không lưu overfit checkpoint.

### Thời gian dự kiến

4–15 phút trên Tesla T4.

### Tiến độ hiện tại

Cell 9/11 — kiểm tra khả năng học và độ an toàn FP16.

### Đầu ra dự kiến

```text
Status                    : PASSED
Final token accuracy      : >= 0.90
Final loss ratio          : <= 0.25
Overflow retries          : <= 10
Pristine model restored   : True
Progress                  : 9 / 11 cells passed
```


In [9]:
CELL_STARTED_AT = time.perf_counter()
required_from_cell8 = [
    "MODEL",
    "TRAIN_SMOKE_SAMPLES",
    "BLIP_GENERATION_FEATURE_CONTRACT_PATH",
]
missing = [name for name in required_from_cell8 if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–8: {missing}"


def create_grad_scaler():
    try:
        return torch.amp.GradScaler(
            "cuda", init_scale=1024.0, growth_interval=2000
        )
    except (TypeError, AttributeError):
        return torch.cuda.amp.GradScaler(
            init_scale=1024.0, growth_interval=2000
        )


def model_parameter_groups(model, vision_lr, text_lr):
    vision_parameters = [
        parameter
        for parameter in model.vision_model.parameters()
        if parameter.requires_grad
    ]
    vision_ids = {id(parameter) for parameter in vision_parameters}
    text_parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad and id(parameter) not in vision_ids
    ]
    assert vision_parameters and text_parameters
    assert not (
        {id(parameter) for parameter in vision_parameters}
        & {id(parameter) for parameter in text_parameters}
    )
    return [
        {"params": vision_parameters, "lr": vision_lr, "name": "vision"},
        {"params": text_parameters, "lr": text_lr, "name": "text"},
    ]


def shifted_token_accuracy(logits, labels):
    predictions = logits[:, :-1].argmax(dim=-1)
    targets = labels[:, 1:]
    valid = targets.ne(-100)
    if not valid.any():
        return 0.0
    return float(
        predictions[valid].eq(targets[valid]).float().mean().item()
    )


OVERFIT_BATCH = encode_samples(TRAIN_SMOKE_SAMPLES[:2])
overfit_labels = OVERFIT_BATCH["input_ids"].clone()
overfit_labels[
    overfit_labels == BLIP_TOKENIZER.pad_token_id
] = -100


def evaluate_overfit(model):
    model.eval()
    with torch.inference_mode(), torch.autocast(
        device_type="cuda", dtype=torch.float16
    ):
        output = model(
            pixel_values=OVERFIT_BATCH["pixel_values"],
            input_ids=OVERFIT_BATCH["input_ids"],
            attention_mask=OVERFIT_BATCH["attention_mask"],
            labels=overfit_labels,
            return_dict=True,
        )
    return (
        float(output.loss.item()),
        shifted_token_accuracy(output.logits, overfit_labels),
    )


set_all_seeds(SEED)
MODEL.train()
OVERFIT_OPTIMIZER = torch.optim.AdamW(
    model_parameter_groups(
        MODEL,
        vision_lr=5e-5,
        text_lr=3e-4,
    ),
    betas=(0.9, 0.999),
    eps=1e-8,
    weight_decay=0.0,
)
OVERFIT_SCALER = create_grad_scaler()
initial_loss, initial_accuracy = evaluate_overfit(MODEL)
curve_rows = []
completed_steps = 0
attempts = 0
overflow_retries = 0
MAX_OVERFIT_STEPS = 120
MIN_OVERFIT_STEPS = 20
torch.cuda.reset_peak_memory_stats()

while completed_steps < MAX_OVERFIT_STEPS:
    attempts += 1
    MODEL.train()
    OVERFIT_OPTIMIZER.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        output = MODEL(
            pixel_values=OVERFIT_BATCH["pixel_values"],
            input_ids=OVERFIT_BATCH["input_ids"],
            attention_mask=OVERFIT_BATCH["attention_mask"],
            labels=overfit_labels,
            return_dict=True,
        )
        loss = output.loss
    assert torch.isfinite(loss)
    old_scale = float(OVERFIT_SCALER.get_scale())
    OVERFIT_SCALER.scale(loss).backward()
    OVERFIT_SCALER.unscale_(OVERFIT_OPTIMIZER)
    gradient_norm = float(
        torch.nn.utils.clip_grad_norm_(MODEL.parameters(), 1.0).item()
    )
    assert math.isfinite(gradient_norm)
    OVERFIT_SCALER.step(OVERFIT_OPTIMIZER)
    OVERFIT_SCALER.update()
    new_scale = float(OVERFIT_SCALER.get_scale())
    if new_scale < old_scale:
        overflow_retries += 1
        assert overflow_retries <= 10
        continue
    completed_steps += 1
    accuracy = shifted_token_accuracy(output.logits, overfit_labels)
    curve_rows.append(
        {
            "step": completed_steps,
            "loss": float(loss.detach().item()),
            "token_accuracy": accuracy,
            "gradient_norm": gradient_norm,
            "grad_scaler_scale": new_scale,
        }
    )
    if completed_steps == 1 or completed_steps % 10 == 0:
        print(
            f"step={completed_steps:03d} | "
            f"loss={float(loss.detach().item()):.6f} | "
            f"token_acc={accuracy:.4f} | "
            f"grad={gradient_norm:.4f}"
        )
    if completed_steps >= MIN_OVERFIT_STEPS:
        eval_loss, eval_accuracy = evaluate_overfit(MODEL)
        if (
            eval_accuracy >= 0.90
            and eval_loss / initial_loss <= 0.25
        ):
            break

final_loss, final_accuracy = evaluate_overfit(MODEL)
final_loss_ratio = final_loss / initial_loss
assert final_accuracy >= 0.90, final_accuracy
assert final_loss_ratio <= 0.25, final_loss_ratio
assert overflow_retries <= 10
peak_gpu_mib = torch.cuda.max_memory_allocated() / 1024**2

curve_path = LOG_DIR / "teacher_b_one_batch_overfit_curve.csv"
with open(curve_path, "w", newline="", encoding="utf-8") as destination:
    writer = csv.DictWriter(
        destination,
        fieldnames=list(curve_rows[0].keys()),
    )
    writer.writeheader()
    writer.writerows(curve_rows)

del OVERFIT_OPTIMIZER, OVERFIT_SCALER, output
del MODEL
gc.collect()
torch.cuda.empty_cache()
MODEL = load_blip_model()
MODEL.eval()
source_weight_sha_after = sha256_file(BLIP_WEIGHT_FILES[0])
source_weight_sha_expected = next(
    item["sha256"]
    for item in BLIP_SOURCE_MANIFEST["files"]
    if item["relative_path"]
    == BLIP_WEIGHT_FILES[0].relative_to(
        BLIP_SOURCE_DIR
    ).as_posix()
)
pristine_source_unchanged = (
    source_weight_sha_after == source_weight_sha_expected
)
assert pristine_source_unchanged

cell_audit_path = (
    AUDIT_DIR / "phase3f_cell09_one_batch_overfit_fp16.json"
)
write_json(
    cell_audit_path,
    {
        "schema_version": "phase3f_cell09_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "distinct_train_images": 2,
        "augmentation": "disabled",
        "maximum_steps": MAX_OVERFIT_STEPS,
        "minimum_steps": MIN_OVERFIT_STEPS,
        "completed_steps": completed_steps,
        "attempts": attempts,
        "initial_loss": initial_loss,
        "final_loss": final_loss,
        "final_loss_ratio": final_loss_ratio,
        "initial_token_accuracy": initial_accuracy,
        "final_token_accuracy": final_accuracy,
        "overflow_retries": overflow_retries,
        "overflow_retry_limit": 10,
        "peak_gpu_mib": peak_gpu_mib,
        "curve": str(curve_path),
        "curve_sha256": sha256_file(curve_path),
        "overfit_checkpoint_saved": False,
        "pristine_model_reloaded": True,
        "pristine_source_unchanged": pristine_source_unchanged,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03F CELL 9 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Distinct train images     : 2")
print("Completed steps           :", completed_steps)
print("Initial loss              :", initial_loss)
print("Final loss                :", final_loss)
print("Final loss ratio          :", final_loss_ratio)
print("Final token accuracy      :", final_accuracy)
print("Overflow retries          :", overflow_retries, "/ 10")
print("Peak GPU memory           :", round(peak_gpu_mib, 2), "MiB")
print("Overfit checkpoint saved  : False")
print("Pristine model restored   :", pristine_source_unchanged)
print("Test Set status           : SEALED")
print("Progress                  : 9 / 11 cells passed")
print("Next                      : exact-resume and training contract")


step=001 | loss=2.657181 | token_acc=0.5417 | grad=24.4140
step=010 | loss=0.348603 | token_acc=0.9583 | grad=1.2790
step=020 | loss=0.035399 | token_acc=1.0000 | grad=1.1666


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: /kaggle/input/models/nguynthnhdan/blip-image-captioning-base-offline/pytorch/82a37760-pytorch/1/salesforce_blip_image_captioning_base_82a37760
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:

PHASE 03F CELL 9 STATUS
Status                    : PASSED
Distinct train images     : 2
Completed steps           : 20
Initial loss              : 2.6571807861328125
Final loss                : 0.0211322158575058
Final loss ratio          : 0.007952870940430457
Final token accuracy      : 1.0
Overflow retries          : 0 / 10
Peak GPU memory           : 5213.48 MiB
Overfit checkpoint saved  : False
Pristine model restored   : True
Test Set status           : SEALED
Progress                  : 9 / 11 cells passed
Next                      : exact-resume and training contract


## Cell 10 — Exact-resume và full-training contract

### Công việc

- so sánh nhánh chạy liên tục 2 step với nhánh 1 + save/load + 1 step;
- so sánh bitwise model, optimizer, scheduler, scaler và RNG;
- xóa checkpoint tạm;
- khóa contract full training seed 42;
- tiếp tục giữ Test niêm phong.

### Thời gian dự kiến

5–20 phút; checkpoint tạm có thể đạt vài GB nhưng sẽ được xóa.

### Tiến độ hiện tại

Cell 10/11 — chứng minh khả năng resume và khóa protocol huấn luyện.

### Đầu ra dự kiến

```text
Status                    : PASSED
Exact-resume              : PASSED
Temporary checkpoint      : DELETED
Training contract         : LOCKED
Full training seed        : 42
Progress                  : 10 / 11 cells passed
```


In [10]:
CELL_STARTED_AT = time.perf_counter()
required_from_cell9 = [
    "MODEL",
    "create_grad_scaler",
    "model_parameter_groups",
]
missing = [name for name in required_from_cell9 if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–9: {missing}"

EXACT_BATCH = encode_samples(TRAIN_SMOKE_SAMPLES[:2])
exact_labels = EXACT_BATCH["input_ids"].clone()
exact_labels[exact_labels == BLIP_TOKENIZER.pad_token_id] = -100


def make_exact_runtime():
    model = load_blip_model()
    model.train()
    optimizer = torch.optim.AdamW(
        model_parameter_groups(
            model,
            vision_lr=1e-5,
            text_lr=5e-5,
        ),
        betas=(0.9, 0.999),
        eps=1e-8,
        weight_decay=0.05,
    )
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda step: min((step + 1) / 10.0, 1.0),
    )
    scaler = create_grad_scaler()
    return model, optimizer, scheduler, scaler


def exact_training_step(model, optimizer, scheduler, scaler):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        output = model(
            pixel_values=EXACT_BATCH["pixel_values"],
            input_ids=EXACT_BATCH["input_ids"],
            attention_mask=EXACT_BATCH["attention_mask"],
            labels=exact_labels,
            return_dict=True,
        )
        loss = output.loss
    assert torch.isfinite(loss)
    old_scale = float(scaler.get_scale())
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    gradient_norm = float(
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0).item()
    )
    assert math.isfinite(gradient_norm)
    scaler.step(optimizer)
    scaler.update()
    new_scale = float(scaler.get_scale())
    assert new_scale >= old_scale, (
        "Exact-resume probe không cho phép overflow."
    )
    scheduler.step()
    return float(loss.detach().item()), gradient_norm


def digest_object(value):
    digest = hashlib.sha256()
    def update(item):
        if isinstance(item, torch.Tensor):
            tensor = item.detach().cpu().contiguous()
            digest.update(b"tensor")
            digest.update(str(tensor.dtype).encode())
            digest.update(str(tuple(tensor.shape)).encode())
            digest.update(tensor.numpy().tobytes())
        elif isinstance(item, np.ndarray):
            digest.update(b"ndarray")
            digest.update(str(item.dtype).encode())
            digest.update(str(item.shape).encode())
            digest.update(item.tobytes())
        elif isinstance(item, dict):
            digest.update(b"dict")
            for key in sorted(item, key=lambda candidate: repr(candidate)):
                update(key)
                update(item[key])
        elif isinstance(item, (list, tuple)):
            digest.update(type(item).__name__.encode())
            for child in item:
                update(child)
        else:
            digest.update(type(item).__name__.encode())
            digest.update(repr(item).encode())
    update(value)
    return digest.hexdigest()


def capture_rng_states():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch_cpu": torch.get_rng_state(),
        "torch_cuda": torch.cuda.get_rng_state_all(),
    }


def restore_rng_states(states):
    random.setstate(states["python"])
    np.random.set_state(states["numpy"])
    torch.set_rng_state(states["torch_cpu"])
    torch.cuda.set_rng_state_all(states["torch_cuda"])


def atomic_torch_save(payload, destination):
    destination = Path(destination)
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    torch.save(payload, temporary)
    os.replace(temporary, destination)


TEMPORARY_RESUME_CHECKPOINT = (
    OUTPUT_ROOT / "_temporary_blip_exact_resume.pt"
)
if TEMPORARY_RESUME_CHECKPOINT.exists():
    TEMPORARY_RESUME_CHECKPOINT.unlink()

del MODEL
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Continuous branch: two optimizer steps.
set_all_seeds(SEED)
model_a, optimizer_a, scheduler_a, scaler_a = make_exact_runtime()
loss_a1, grad_a1 = exact_training_step(
    model_a, optimizer_a, scheduler_a, scaler_a
)
loss_a2, grad_a2 = exact_training_step(
    model_a, optimizer_a, scheduler_a, scaler_a
)
continuous_digests = {
    "model": digest_object(model_a.state_dict()),
    "optimizer": digest_object(optimizer_a.state_dict()),
    "scheduler": digest_object(scheduler_a.state_dict()),
    "scaler": digest_object(scaler_a.state_dict()),
    "rng": digest_object(capture_rng_states()),
}
del model_a, optimizer_a, scheduler_a, scaler_a
gc.collect()
torch.cuda.empty_cache()

# Resume branch: one step, checkpoint, reconstruct, load, one step.
set_all_seeds(SEED)
model_b, optimizer_b, scheduler_b, scaler_b = make_exact_runtime()
loss_b1, grad_b1 = exact_training_step(
    model_b, optimizer_b, scheduler_b, scaler_b
)
assert loss_b1 == loss_a1
assert grad_b1 == grad_a1
atomic_torch_save(
    {
        "schema_version": "phase3f_blip_exact_resume_probe_v1",
        "trusted_local_temporary_checkpoint": True,
        "model": model_b.state_dict(),
        "optimizer": optimizer_b.state_dict(),
        "scheduler": scheduler_b.state_dict(),
        "scaler": scaler_b.state_dict(),
        "rng": capture_rng_states(),
        "progress": {"optimizer_step": 1},
    },
    TEMPORARY_RESUME_CHECKPOINT,
)
temporary_checkpoint_bytes = TEMPORARY_RESUME_CHECKPOINT.stat().st_size
del model_b, optimizer_b, scheduler_b, scaler_b
gc.collect()
torch.cuda.empty_cache()

model_b, optimizer_b, scheduler_b, scaler_b = make_exact_runtime()
try:
    resume_payload = torch.load(
        TEMPORARY_RESUME_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )
except TypeError:
    resume_payload = torch.load(
        TEMPORARY_RESUME_CHECKPOINT,
        map_location="cpu",
    )
assert resume_payload["trusted_local_temporary_checkpoint"] is True
assert resume_payload["progress"]["optimizer_step"] == 1
model_b.load_state_dict(resume_payload["model"], strict=True)
optimizer_b.load_state_dict(resume_payload["optimizer"])
scheduler_b.load_state_dict(resume_payload["scheduler"])
scaler_b.load_state_dict(resume_payload["scaler"])
restore_rng_states(resume_payload["rng"])
del resume_payload
gc.collect()

loss_b2, grad_b2 = exact_training_step(
    model_b, optimizer_b, scheduler_b, scaler_b
)
resume_digests = {
    "model": digest_object(model_b.state_dict()),
    "optimizer": digest_object(optimizer_b.state_dict()),
    "scheduler": digest_object(scheduler_b.state_dict()),
    "scaler": digest_object(scaler_b.state_dict()),
    "rng": digest_object(capture_rng_states()),
}
exact_resume = (
    loss_b2 == loss_a2
    and grad_b2 == grad_a2
    and resume_digests == continuous_digests
)
assert exact_resume, {
    "continuous": continuous_digests,
    "resume": resume_digests,
    "loss": [loss_a2, loss_b2],
    "gradient": [grad_a2, grad_b2],
}

del model_b, optimizer_b, scheduler_b, scaler_b
gc.collect()
torch.cuda.empty_cache()
TEMPORARY_RESUME_CHECKPOINT.unlink()
assert not TEMPORARY_RESUME_CHECKPOINT.exists()

MODEL = load_blip_model()
MODEL.eval()
peak_gpu_mib = torch.cuda.max_memory_allocated() / 1024**2

TRAINING_CONTRACT = {
    "contract_version": "phase3_teacher_b_full_training_v1",
    "created_at_utc": now_utc(),
    "status": "locked",
    "teacher_id": "B",
    "model": EXPECTED_REPO_ID,
    "resolved_revision": EXPECTED_REVISION,
    "initial_full_training_seed": 42,
    "confirmation_seeds_after_seed42_gate": [123, 2026],
    "dataset": "MS COCO 2014 only",
    "split_protocol": "Karpathy",
    "training_images_per_epoch": 113287,
    "captions_per_image_per_epoch": 1,
    "sampling": "deterministic image-balanced cyclic caption",
    "image_size": 384,
    "maximum_text_length": MAX_TEXT_LENGTH,
    "physical_batch_size": 2,
    "gradient_accumulation_steps": 16,
    "effective_batch_size": 32,
    "maximum_epochs": 10,
    "optimizer": {
        "name": "AdamW",
        "vision_learning_rate": 1e-5,
        "text_learning_rate": 5e-5,
        "betas": [0.9, 0.999],
        "epsilon": 1e-8,
        "weight_decay": 0.05,
    },
    "scheduler": {
        "name": "linear warmup plus cosine decay",
        "warmup_fraction": 0.05,
        "minimum_lr_factor": 0.05,
    },
    "mixed_precision": "FP16 plus GradScaler",
    "grad_scaler_initial_scale": 1024.0,
    "overflow_retry_limit": 10,
    "gradient_clip_norm": 1.0,
    "validation": {
        "images": 5000,
        "references": 25010,
        "teacher_forced_ce": True,
        "generation": {
            "beam_size": 3,
            "length_penalty": 0.7,
            "maximum_new_tokens": 31,
        },
        "primary_selection_metric": "CIDEr-D",
        "tie_break_order": [
            "BLEU-4 higher",
            "Validation CE lower",
            "earlier epoch",
        ],
        "minimum_seed42_cider_d": phase3e_auth["entry_contract"]
        and read_json(
            PHASE3E_ROOT
            / phase3e_auth["entry_contract"]["relative_path"]
        )["validation"]["quality_targets_by_seed"]["42"][
            "teacher_minimum_validation_cider_d"
        ],
    },
    "early_stopping": {
        "minimum_epoch": 3,
        "patience": 3,
        "minimum_cider_delta": 0.001,
    },
    "kd_policy": {
        "direct_logit_kd": "prohibited",
        "feature_kd": "allowed after projection contract",
        "sequence_kd": "allowed",
    },
    "checkpoint_boundary": "optimizer-step and epoch boundary",
    "atomic_checkpoint": True,
    "exact_resume_required": True,
    "test_set_status": "sealed",
    "test_set_used": False,
}
BLIP_TRAINING_CONTRACT_PATH = (
    CONFIG_DIR / "teacher_b_full_training_contract.json"
)
write_json(BLIP_TRAINING_CONTRACT_PATH, TRAINING_CONTRACT)

BLIP_EXACT_RESUME_CONTRACT_PATH = (
    CONFIG_DIR / "teacher_b_exact_resume_contract.json"
)
write_json(
    BLIP_EXACT_RESUME_CONTRACT_PATH,
    {
        "contract_version": "phase3_teacher_b_exact_resume_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "comparison": "continuous 2 steps vs 1 + save/load + 1",
        "batch_size": 2,
        "model_exact": (
            continuous_digests["model"] == resume_digests["model"]
        ),
        "optimizer_exact": (
            continuous_digests["optimizer"]
            == resume_digests["optimizer"]
        ),
        "scheduler_exact": (
            continuous_digests["scheduler"]
            == resume_digests["scheduler"]
        ),
        "grad_scaler_exact": (
            continuous_digests["scaler"]
            == resume_digests["scaler"]
        ),
        "rng_exact": (
            continuous_digests["rng"] == resume_digests["rng"]
        ),
        "loss_exact": loss_a2 == loss_b2,
        "gradient_norm_exact": grad_a2 == grad_b2,
        "temporary_checkpoint_bytes": temporary_checkpoint_bytes,
        "temporary_checkpoint_deleted": True,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

cell_audit_path = (
    AUDIT_DIR / "phase3f_cell10_exact_resume_training_contract.json"
)
write_json(
    cell_audit_path,
    {
        "schema_version": "phase3f_cell10_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "continuous_losses": [loss_a1, loss_a2],
        "resume_losses": [loss_b1, loss_b2],
        "continuous_gradient_norms": [grad_a1, grad_a2],
        "resume_gradient_norms": [grad_b1, grad_b2],
        "continuous_digests": continuous_digests,
        "resume_digests": resume_digests,
        "exact_resume": exact_resume,
        "temporary_checkpoint_bytes": temporary_checkpoint_bytes,
        "temporary_checkpoint_deleted": True,
        "peak_gpu_mib": peak_gpu_mib,
        "exact_resume_contract": str(
            BLIP_EXACT_RESUME_CONTRACT_PATH
        ),
        "training_contract": str(BLIP_TRAINING_CONTRACT_PATH),
        "full_training_started": False,
        "test_parquet_opened": False,
        "test_images_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03F CELL 10 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Continuous losses         :", [loss_a1, loss_a2])
print("Resume losses             :", [loss_b1, loss_b2])
print("Model state exact         : True")
print("Optimizer state exact     : True")
print("Scheduler state exact     : True")
print("GradScaler state exact    : True")
print("RNG states exact          : True")
print("Temporary checkpoint bytes:", temporary_checkpoint_bytes)
print("Temporary checkpoint      : DELETED")
print("Training contract         : LOCKED")
print("Physical/effective batch  : 2 / 32")
print("Full training seed        : 42")
print("Full training started     : False")
print("Test Set status           : SEALED")
print("Progress                  : 10 / 11 cells passed")
print("Next                      : package Phase 03F")


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: /kaggle/input/models/nguynthnhdan/blip-image-captioning-base-offline/pytorch/82a37760-pytorch/1/salesforce_blip_image_captioning_base_82a37760
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: /kaggle/input/models/nguynthnhdan/blip-image-captioning-base-offline/pytorch/82a37760-pytorch/1/salesforce_blip_image_captioning_base_82a37760
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: /kaggle/input/models/nguynthnhdan/blip-image-captioning-base-offline/pytorch/82a37760-pytorch/1/salesforce_blip_image_captioning_base_82a37760
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: /kaggle/input/models/nguynthnhdan/blip-image-captioning-base-offline/pytorch/82a37760-pytorch/1/salesforce_blip_image_captioning_base_82a37760
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:

PHASE 03F CELL 10 STATUS
Status                    : PASSED
Continuous losses         : [2.6571807861328125, 2.323491334915161]
Resume losses             : [2.6571807861328125, 2.323491334915161]
Model state exact         : True
Optimizer state exact     : True
Scheduler state exact     : True
GradScaler state exact    : True
RNG states exact          : True
Temporary checkpoint bytes: 2969751267
Temporary checkpoint      : DELETED
Training contract         : LOCKED
Physical/effective batch  : 2 / 32
Full training seed        : 42
Full training started     : False
Test Set status           : SEALED
Progress                  : 10 / 11 cells passed
Next                      : package Phase 03F


## Cell 11 — Package Phase 03F và phát hành full-training authorization

### Công việc

- xác minh 10/10 cell audit và mọi locked contract;
- tạo manifest CSV/JSON/SHA256SUMS;
- phát hành BLIP preflight COMPLETE marker;
- chỉ phát hành full-training AUTHORIZED khi mọi gate PASS;
- không lưu diagnostic weights và không mở Test.

### Thời gian dự kiến

10–60 giây.

### Tiến độ hiện tại

Cell 11/11 — đóng gói và kết thúc Phase 03F.

### Đầu ra bắt buộc

```text
Status                    : COMPLETE
Teacher B BLIP preflight  : VERIFIED
Full training seed 42     : AUTHORIZED
Manifest failures         : 0
Test Set status           : SEALED
Progress                  : 11 / 11 cells passed
```


In [11]:
CELL_STARTED_AT = time.perf_counter()
required_from_cell10 = [
    "BLIP_TRAINING_CONTRACT_PATH",
    "BLIP_EXACT_RESUME_CONTRACT_PATH",
    "TRAINING_CONTRACT",
    "MODEL",
]
missing = [name for name in required_from_cell10 if name not in globals()]
assert not missing, f"Thiếu biến Cell 1–10: {missing}"

required_audits = [
    "phase3f_cell01_environment_discovery.json",
    "phase3f_cell02_upstream_contract_integrity.json",
    "phase3f_cell03_blip_source_integrity.json",
    "phase3f_cell04_processor_tokenizer.json",
    "phase3f_cell05_development_data_token_length.json",
    "phase3f_cell06_model_architecture_forward.json",
    "phase3f_cell07_loss_backward_smoke.json",
    "phase3f_cell08_generation_determinism_feature.json",
    "phase3f_cell09_one_batch_overfit_fp16.json",
    "phase3f_cell10_exact_resume_training_contract.json",
]
for name in required_audits:
    path = AUDIT_DIR / name
    assert path.is_file(), path
    payload = read_json(path)
    assert payload["status"] == "passed", name
    assert payload["test_set_status"] == "sealed", name
    assert payload["test_set_used"] is False, name

required_contracts = [
    BLIP_SOURCE_CONTRACT_PATH,
    BLIP_PROCESSOR_CONTRACT_PATH,
    BLIP_SEQUENCE_CONTRACT_PATH,
    BLIP_ARCHITECTURE_CONTRACT_PATH,
    BLIP_GENERATION_FEATURE_CONTRACT_PATH,
    BLIP_EXACT_RESUME_CONTRACT_PATH,
    BLIP_TRAINING_CONTRACT_PATH,
]
for path in required_contracts:
    assert path.is_file(), path
    payload = read_json(path)
    assert payload["status"] in {"locked", "passed"}, path
    assert payload["test_set_status"] == "sealed", path
    assert payload["test_set_used"] is False, path

del MODEL
gc.collect()
torch.cuda.empty_cache()

artifact_paths = sorted(
    [
        path
        for parent in [AUDIT_DIR, CONFIG_DIR, LOG_DIR]
        for path in parent.rglob("*")
        if path.is_file()
    ]
)
manifest_rows = [
    {
        "relative_path": path.relative_to(OUTPUT_ROOT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
]
assert len(manifest_rows) == 18, len(manifest_rows)

MANIFEST_CSV_PATH = MANIFEST_DIR / "phase3f_manifest.csv"
with open(MANIFEST_CSV_PATH, "w", newline="", encoding="utf-8") as target:
    writer = csv.DictWriter(
        target,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)

MANIFEST_JSON_PATH = MANIFEST_DIR / "phase3f_manifest.json"
write_json(
    MANIFEST_JSON_PATH,
    {
        "schema_version": "phase3f_manifest_v1",
        "created_at_utc": now_utc(),
        "artifact_count": len(manifest_rows),
        "artifacts": manifest_rows,
    },
)
SHA256SUMS_PATH = MANIFEST_DIR / "phase3f_SHA256SUMS.txt"
SHA256SUMS_PATH.write_text(
    "".join(
        f"{row['sha256']}  {row['relative_path']}\n"
        for row in manifest_rows
    ),
    encoding="utf-8",
)

manifest_failures = []
for row in manifest_rows:
    path = (OUTPUT_ROOT / row["relative_path"]).resolve()
    assert path.is_relative_to(OUTPUT_ROOT.resolve())
    if not path.is_file():
        manifest_failures.append(
            {"relative_path": row["relative_path"], "reason": "missing"}
        )
    elif path.stat().st_size != row["size_bytes"]:
        manifest_failures.append(
            {"relative_path": row["relative_path"], "reason": "size"}
        )
    elif sha256_file(path) != row["sha256"]:
        manifest_failures.append(
            {"relative_path": row["relative_path"], "reason": "sha256"}
        )
assert not manifest_failures, manifest_failures

manifest_refs = {
    "csv": {
        "relative_path": MANIFEST_CSV_PATH.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "sha256": sha256_file(MANIFEST_CSV_PATH),
    },
    "json": {
        "relative_path": MANIFEST_JSON_PATH.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "sha256": sha256_file(MANIFEST_JSON_PATH),
    },
    "sha256sums": {
        "relative_path": SHA256SUMS_PATH.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "sha256": sha256_file(SHA256SUMS_PATH),
    },
}

PREFLIGHT_COMPLETE_PATH = (
    OUTPUT_ROOT / "PHASE3_TEACHER_B_BLIP_PREFLIGHT_COMPLETE.json"
)
write_json(
    PREFLIGHT_COMPLETE_PATH,
    {
        "marker_version": "phase3_teacher_b_blip_preflight_complete_v1",
        "created_at_utc": now_utc(),
        "status": "complete",
        "teacher_id": "B",
        "model": EXPECTED_REPO_ID,
        "resolved_revision": EXPECTED_REVISION,
        "source_integrity": "passed",
        "processor_tokenizer_contract": "locked",
        "native_sequence_contract": "locked",
        "architecture_forward": "passed",
        "loss_backward": "passed",
        "generation_determinism": "passed",
        "one_batch_overfit": "passed",
        "fp16_preflight": "passed",
        "exact_resume": "passed",
        "direct_logit_kd": "prohibited",
        "feature_kd": "allowed_after_projection_contract",
        "sequence_kd": "allowed",
        "diagnostic_weights_saved": False,
        "full_training_started": False,
        "training_contract": {
            "relative_path": BLIP_TRAINING_CONTRACT_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(BLIP_TRAINING_CONTRACT_PATH),
        },
        "manifests": manifest_refs,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

FULL_TRAINING_AUTHORIZED_PATH = (
    OUTPUT_ROOT / "PHASE3_TEACHER_B_SEED42_FULL_TRAINING_AUTHORIZED.json"
)
write_json(
    FULL_TRAINING_AUTHORIZED_PATH,
    {
        "marker_version": (
            "phase3_teacher_b_seed42_full_training_authorized_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "ready",
        "teacher_id": "B",
        "model": EXPECTED_REPO_ID,
        "resolved_revision": EXPECTED_REVISION,
        "seed": 42,
        "preflight_complete": {
            "relative_path": PREFLIGHT_COMPLETE_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(PREFLIGHT_COMPLETE_PATH),
        },
        "training_contract": {
            "relative_path": BLIP_TRAINING_CONTRACT_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(BLIP_TRAINING_CONTRACT_PATH),
        },
        "manifests": manifest_refs,
        "authorized_next_action": (
            "Create Phase 03G Teacher B BLIP seed-42 full training "
            "notebook; keep Held-out Test sealed."
        ),
        "full_training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

assert read_json(PREFLIGHT_COMPLETE_PATH)["status"] == "complete"
assert read_json(FULL_TRAINING_AUTHORIZED_PATH)["status"] == "ready"
assert read_json(FULL_TRAINING_AUTHORIZED_PATH)["test_set_used"] is False

print("=" * 72)
print("PHASE 03F CELL 11 STATUS")
print("=" * 72)
print("Status                    : COMPLETE")
print("Teacher B BLIP preflight  : VERIFIED")
print("Repository                :", EXPECTED_REPO_ID)
print("Resolved revision         :", EXPECTED_REVISION)
print("One-batch overfit         : PASSED")
print("FP16 preflight            : PASSED")
print("Exact-resume              : PASSED")
print("Training contract         : LOCKED")
print("Full training seed 42     : AUTHORIZED")
print("Manifest rows             :", len(manifest_rows))
print("Manifest failures         :", len(manifest_failures))
print("Diagnostic weights saved  : False")
print("Full training started     : False")
print("Test Parquet opened       : False")
print("Test metrics computed     : False")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Progress                  : 11 / 11 cells passed")
print("Next                      : Phase 03G Teacher B seed-42 full training")


PHASE 03F CELL 11 STATUS
Status                    : COMPLETE
Teacher B BLIP preflight  : VERIFIED
Repository                : Salesforce/blip-image-captioning-base
Resolved revision         : 82a37760796d32b1411fe092ab5d4e227313294b
One-batch overfit         : PASSED
FP16 preflight            : PASSED
Exact-resume              : PASSED
Training contract         : LOCKED
Full training seed 42     : AUTHORIZED
Manifest rows             : 18
Manifest failures         : 0
Diagnostic weights saved  : False
Full training started     : False
Test Parquet opened       : False
Test metrics computed     : False
Test Set status           : SEALED
Test Set used             : False
Progress                  : 11 / 11 cells passed
Next                      : Phase 03G Teacher B seed-42 full training
